# 03 - CC-MVAD multi-view actor discovery

Community-Conditioned Multi-View Actor Discovery combines graph community context with behavioural deviation. It is unsupervised during representation construction.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.actor_features import make_actor_dataset
from src.clustering import prepare_matrix
from src.evaluation import cluster_enrichment
from src.multiview import community_conditioned_residuals, robust_zscore
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'
sns.set_theme(style='whitegrid')

In [ ]:
actor = make_actor_dataset(DATA / 'Actors_Dataset' / 'wallets_features_classes_combined.csv', DATA / 'Actors_Dataset' / 'wallets_classes.csv')
sample = actor.sample(n=min(20000, len(actor)), random_state=42).copy()
embedding, pca, _, _ = prepare_matrix(sample, random_state=42)
memberships = pd.read_csv(RESULTS / 'leiden_memberships.csv', index_col='address')['community']
memberships.index = memberships.index.astype(str)
sample['community'] = memberships.reindex(sample.index.astype(str)).to_numpy()
valid = sample['community'].notna().to_numpy()
sample['behavioral_residual'] = np.nan
sample.loc[valid, 'behavioral_residual'], community_stats = community_conditioned_residuals(embedding[valid], sample.loc[valid, 'community'].to_numpy())
sample['residual_z'] = robust_zscore(sample['behavioral_residual'].fillna(0).to_numpy())
print(f'PCA variance: {pca.explained_variance_ratio_.sum():.4f}')
display(community_stats.head(15))

In [ ]:
evaluation = cluster_enrichment(sample['community'].fillna(-1).to_numpy(), sample['class']).reset_index().rename(columns={'cluster': 'community'})
evaluation = evaluation.merge(community_stats, on='community', how='left')
evaluation = evaluation[evaluation['cluster_size'] >= 30].sort_values('enrichment', ascending=False)
display(evaluation.head(15))
evaluation.to_csv(RESULTS / 'cc_mvad_community_evaluation.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.boxplot(data=sample, x='class', y='behavioral_residual', showfliers=False, ax=axes[0])
axes[0].set_title('Behavioural residual by label')
sns.scatterplot(data=evaluation, x='mean_behavioral_residual', y='enrichment', size='cluster_size', hue='illicit_rate', sizes=(30, 500), palette='magma', legend=False, ax=axes[1])
axes[1].set_title('CC-MVAD community evaluation')
fig.tight_layout()
fig.savefig(FIGURES / 'cc_mvad_community_evaluation.png', dpi=180, bbox_inches='tight')
plt.show()